# Validation of `LinearSmallStrainFEMForceField`

This notebook validates SOFA's `LinearSmallStrainFEMForceField` against FEniCS, as presented in [IntroductionValidation.ipynb](IntroductionValidation.ipynb), on the cases in `validation/cases/LinearSmallStrainFEMForceField/`. Where a case has an analytical solution, SOFA's solution is compared with it as well.

## Running the Cases

The cells below load the suite's runner and prepare the figures. Run them once before any case: every case cell below then runs on its own, in any order.

In [ ]:
# Make the plugin's packages importable from validation/docs
import sys
sys.path.insert(0, "../..")

# Hide the runner's printed report; figures still show
import os
sys.stdout = open(os.devnull, "w")

# Load the suite's runner
from validation.run import run

# For showing figures inline
%matplotlib inline

`side_by_side` shows the figures of a case's elements in one row.

In [ ]:
import base64
import io
from IPython.display import HTML, display

def side_by_side(*figures):
    images = []
    for figure in figures:
        buffer = io.BytesIO()
        figure.savefig(buffer, format="png")
        encoded = base64.b64encode(buffer.getvalue()).decode()
        images.append(f'<img src="data:image/png;base64,{encoded}" style="width: {100 / len(figures)}%">')
    display(HTML(f'<div style="display: flex">{"".join(images)}</div>'))

`plot_deformed` draws SOFA's deformed beam for each element, coloured by $|\mathbf{u}|$.

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np

def plot_deformed(study):
    dim = next(iter(study.solutions.values()))[0].shape[1]
    figure, axes = plt.subplots(1, len(study.solutions), figsize=(12, 3 if dim == 2 else 4), layout="constrained",
                                subplot_kw={"projection": "3d"} if dim == 3 else {})
    magnitudes = {element: np.linalg.norm(u, axis=1) for element, (x, u) in study.solutions.items()}
    norm = plt.Normalize(0, max(magnitude.max() for magnitude in magnitudes.values()))
    # The largest displacement is drawn at a tenth of the beam's length
    length = np.ptp(next(iter(study.solutions.values()))[0][:, 0])
    scale = float(f"{0.1 * length / norm.vmax:.1g}")

    # In 3D, y is drawn upwards, since the transverse loads act along it
    def view(points):
        return (points[..., 0], points[..., 1]) if dim == 2 else (points[..., 0], points[..., 2], points[..., 1])

    for ax, (element, (x, u)) in zip(axes, study.solutions.items()):
        # The nodes follow the grid order i + nx*(j + ny*k)
        shape = tuple(len(np.unique(x[:, axis].round(10))) for axis in reversed(range(dim)))
        deformed = (x + scale * u).reshape(*shape, dim)
        magnitude = magnitudes[element].reshape(shape)
        if dim == 2:
            ax.pcolormesh(*view(deformed), magnitude, shading="gouraud", cmap="viridis", norm=norm)
            ax.set_aspect("equal")
        else:
            for face in (np.s_[0], np.s_[-1], np.s_[:, 0], np.s_[:, -1], np.s_[:, :, 0], np.s_[:, :, -1]):
                ax.plot_surface(*view(deformed[face]), facecolors=plt.cm.viridis(norm(magnitude[face])),
                                shade=False)
            ax.set_box_aspect([np.ptp(coordinate) for coordinate in view(deformed)])
            ax.set(xlabel="x", ylabel="z", zlabel="y")

        corners = np.array(list(itertools.product(*zip(x.min(axis=0), x.max(axis=0)))))
        edges = [(a, b) for a, b in itertools.combinations(corners, 2) if np.count_nonzero(a != b) == 1]
        for k, edge in enumerate(edges):
            ax.plot(*view(np.array(edge)), "k--", lw=0.8, label="undeformed" if k == 0 else None)
        ax.set_title(f"{element}, displacement ×{scale:g}")

    figure.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="viridis"), ax=axes, label="|u|")
    figure.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center")
    plt.show()

## Edge Elements on a 1D Bar

The bar starts at the origin and has length $L$ along $x$. The displacement $u$ is a scalar.

### Distributed Axial Load

The bar carries a uniform axial load $q$ per unit length. With Young's modulus $E$, the displacement satisfies $-E\,u'' = q$, and its exact solution is

$$u(x) = \frac{q}{E}\left(L x - \frac{x^2}{2}\right)$$

- `left` ($x = 0$): $u = 0$
- `right` ($x = L$): $t = 0$, traction-free, with nothing applied

In [ ]:
# Case: 1D bar under a distributed axial load
study = run("../cases/LinearSmallStrainFEMForceField/1D/DistributedAxialLoad")
# SOFA, FEniCS and the exact solution along the bar
study.plots["edge"]

## Triangle and Quadrilateral Elements on a 2D Beam

The beam starts at the origin, with length $L$ along $x$ and width $W$ along $y$. Each case is run on quadrilateral (`quad`) and triangle (`tri`) elements.

### Axial Compression

The right side carries a uniform compressive traction $p$. The left side is held only along $x$, so that the beam can contract freely across its width, and the stress is uniform, $\sigma_{xx} = -p$. The exact solution is

$$u_x = -\frac{p}{E}\,x, \qquad u_y = \frac{\nu p}{E}\,y$$

- `left` ($x = 0$): $u_x = 0$
- corner $(0, 0)$: $u_y = 0$
- `right` ($x = L$): $\mathbf{t} = (-p, 0)$
- `bottom` ($y = 0$) and `top` ($y = W$): $\mathbf{t} = 0$

In [ ]:
# Case: 2D beam under axial compression
study = run("../cases/LinearSmallStrainFEMForceField/2D/AxialCompression")
# quad on the left, tri on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["quad"], study.plots["tri"])
# SOFA's deformed beam, coloured by |u|
plot_deformed(study)

### Distributed Transverse Traction

The beam is clamped on its left side, and its top side carries a uniform downward traction $q$. This case has no exact solution, so SOFA is compared with FEniCS only.

- `left` ($x = 0$): $\mathbf{u} = 0$
- `top` ($y = W$): $\mathbf{t} = (0, -q)$
- `right` ($x = L$) and `bottom` ($y = 0$): $\mathbf{t} = 0$

In [ ]:
# Case: 2D cantilever under a distributed transverse traction
study = run("../cases/LinearSmallStrainFEMForceField/2D/DistributedTransverseTraction")
# quad on the left, tri on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["quad"], study.plots["tri"])
# SOFA's deformed beam, coloured by |u|
plot_deformed(study)

### Three-Point Bending

The beam rests on two rollers at $(0, 0)$ and $(L, 0)$, and a point force $P$ pushes down on the middle of its top side. The bottom of the midspan is held along $x$, which stops the beam from sliding without breaking its symmetry. This case has no exact solution, so SOFA is compared with FEniCS only.

- rollers $(0, 0)$ and $(L, 0)$: $u_y = 0$
- node $(L/2, 0)$: $u_x = 0$
- node $(L/2, W)$: force $(0, -P)$
- all sides: $\mathbf{t} = 0$

In [ ]:
# Case: 2D beam in three-point bending
study = run("../cases/LinearSmallStrainFEMForceField/2D/ThreePointBending")
# quad on the left, tri on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["quad"], study.plots["tri"])
# SOFA's deformed beam, coloured by |u|
plot_deformed(study)

## Tetrahedron and Hexahedron Elements on a 3D Beam

The beam starts at the origin, with length $L$ along $x$, width $W$ along $y$ and height $H$ along $z$. Each case is run on hexahedron (`hexa`) and tetrahedron (`tet`) elements. The figures comparing SOFA with FEniCS show the slice $z = H/2$, while $e_\text{rms}$ covers all the nodes.

### Axial Compression

The right face carries a uniform compressive traction $p$. The left face is held only along $x$, so that the beam can contract freely across its section, and the stress is uniform, $\sigma_{xx} = -p$. The exact solution is

$$u_x = -\frac{p}{E}\,x, \qquad u_y = \frac{\nu p}{E}\,y, \qquad u_z = \frac{\nu p}{E}\,z$$

- `left` ($x = 0$): $u_x = 0$
- corner $(0, 0, 0)$: $u_y = u_z = 0$
- corner $(0, W, 0)$: $u_z = 0$
- `right` ($x = L$): $\mathbf{t} = (-p, 0, 0)$
- the other faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D beam under axial compression
study = run("../cases/LinearSmallStrainFEMForceField/3D/AxialCompression")
# hexa on the left, tet on the right, on the slice z = H/2: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["hexa"], study.plots["tet"])
# SOFA's deformed beam, coloured by |u|
plot_deformed(study)

### Distributed Transverse Traction

The beam is clamped on its left face, and its top face carries a uniform downward traction $q$. This case has no exact solution, so SOFA is compared with FEniCS only.

- `left` ($x = 0$): $\mathbf{u} = 0$
- `top` ($y = W$): $\mathbf{t} = (0, -q, 0)$
- the other faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D cantilever under a distributed transverse traction
study = run("../cases/LinearSmallStrainFEMForceField/3D/DistributedTransverseTraction")
# hexa on the left, tet on the right, on the slice z = H/2: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["hexa"], study.plots["tet"])
# SOFA's deformed beam, coloured by |u|
plot_deformed(study)

### Three-Point Bending

The beam rests on a pinned line at $x = 0$ and on a roller line at $x = L$, both on its bottom face and along $z$. A force $P$ pushes down on each node of the line across the middle of its top face. This case has no exact solution, so SOFA is compared with FEniCS only.

- line $(0, 0, z)$: $u_x = u_y = 0$
- line $(L, 0, z)$: $u_y = 0$
- node $(0, 0, 0)$: $u_z = 0$
- line $(L/2, W, z)$: force $(0, -P, 0)$ on each node
- all faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D beam in three-point bending
study = run("../cases/LinearSmallStrainFEMForceField/3D/ThreePointBending")
# hexa on the left, tet on the right, on the slice z = H/2: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["hexa"], study.plots["tet"])
# SOFA's deformed beam, coloured by |u|
plot_deformed(study)